In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import json
import random
import numpy as np
import pandas as pd
import cv2

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("OpenCV:", cv2.__version__)

Mounted at /content/drive
Device: cuda
OpenCV: 4.14.0


In [2]:
import os
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/palm_video_project"

MANIFEST_DIR = os.path.join(
    PROJECT_ROOT,
    "aligned_manifests"
)

EXPERIMENT_ROOT = os.path.join(
    PROJECT_ROOT,
    "experiment_all_original_three_classes"
)

QUALITY_ROOT = os.path.join(
    EXPERIMENT_ROOT,
    "quality_analysis"
)

os.makedirs(EXPERIMENT_ROOT, exist_ok=True)
os.makedirs(QUALITY_ROOT, exist_ok=True)

train_manifest = pd.read_csv(
    os.path.join(
        MANIFEST_DIR,
        "palm_train_manifest_aligned.csv"
    )
)

val_manifest = pd.read_csv(
    os.path.join(
        MANIFEST_DIR,
        "palm_validation_manifest_aligned.csv"
    )
)

test_manifest = pd.read_csv(
    os.path.join(
        MANIFEST_DIR,
        "palm_test_manifest_aligned.csv"
    )
)

for name, df in [
    ("Train", train_manifest),
    ("Validation", val_manifest),
    ("Test", test_manifest)
]:
    print(
        f"{name}: {len(df)} frames, "
        f"{df['ID'].nunique()} patients"
    )
    print(df["Severity_Class"].value_counts(dropna=False))
    print()

Train: 1556 frames, 524 patients
Severity_Class
Normal      1047
Leve         376
Moderado     133
Name: count, dtype: int64

Validation: 276 frames, 92 patients
Severity_Class
Normal      186
Leve         63
Moderado     27
Name: count, dtype: int64

Test: 458 frames, 155 patients
Severity_Class
Normal      309
Leve        108
Moderado     41
Name: count, dtype: int64



In [4]:
import os
import pandas as pd
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/palm_video_project"

ALIGNED_DIR = os.path.join(
    PROJECT_ROOT, "aligned_manifests"
)

ORIGINAL_ROOT = os.path.join(
    PROJECT_ROOT, "video_frame_selection"
)

OUTPUT_ROOT = os.path.join(
    PROJECT_ROOT, "experiment_all_original_three_classes"
)

QUALITY_ROOT = os.path.join(
    OUTPUT_ROOT, "quality_analysis"
)

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(QUALITY_ROOT, exist_ok=True)

# These manifests determine the split assignments only.
train_aligned = pd.read_csv(
    os.path.join(ALIGNED_DIR, "palm_train_manifest_aligned.csv")
)
val_aligned = pd.read_csv(
    os.path.join(ALIGNED_DIR, "palm_validation_manifest_aligned.csv")
)
test_aligned = pd.read_csv(
    os.path.join(ALIGNED_DIR, "palm_test_manifest_aligned.csv")
)

def clean_id(value):
    value = str(value).strip()
    if value.endswith(".0"):
        value = value[:-2]
    return value

train_ids = set(train_aligned["ID"].map(clean_id))
val_ids = set(val_aligned["ID"].map(clean_id))
test_ids = set(test_aligned["ID"].map(clean_id))

print("Train aligned IDs:", len(train_ids))
print("Validation aligned IDs:", len(val_ids))
print("Test aligned IDs:", len(test_ids))

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

assert os.path.isdir(ORIGINAL_ROOT), (
    f"Original image directory not found: {ORIGINAL_ROOT}"
)

# Inspect directory structure.
for root, dirs, files in os.walk(ORIGINAL_ROOT):
    level = root.replace(ORIGINAL_ROOT, "").count(os.sep)

    if level > 3:
        dirs[:] = []
        continue

    print(
        "  " * level,
        os.path.basename(root) or root,
        f"({len(files)} files)"
    )

    for filename in files[:5]:
        print("  " * (level + 1), filename)

Train aligned IDs: 524
Validation aligned IDs: 92
Test aligned IDs: 155
 video_frame_selection (0 files)
   Leve (1 files)
     processing_summary.json
     ID909 (4 files)
       ID909_frame_selection.json
       frame_03.jpg
       frame_01.jpg
       frame_02.jpg
     ID865 (4 files)
       frame_02.jpg
       frame_03.jpg
       ID865_frame_selection.json
       frame_01.jpg
     ID902 (4 files)
       frame_01.jpg
       frame_03.jpg
       frame_02.jpg
       ID902_frame_selection.json
     ID906 (4 files)
       frame_02.jpg
       frame_01.jpg
       frame_03.jpg
       ID906_frame_selection.json
     ID904 (4 files)
       frame_03.jpg
       frame_02.jpg
       ID904_frame_selection.json
       frame_01.jpg
     ID907 (4 files)
       frame_01.jpg
       frame_02.jpg
       ID907_frame_selection.json
       frame_03.jpg
     ID899 (4 files)
       frame_03.jpg
       frame_02.jpg
       frame_01.jpg
       ID899_frame_selection.json
     ID866 (4 files)
       frame_02.jpg
  

In [5]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

original_image_paths = []

for path in Path(ORIGINAL_ROOT).rglob("*"):
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
        original_image_paths.append(str(path))

print("Original image files discovered:", len(original_image_paths))

for path in original_image_paths[:30]:
    print(path)

Original image files discovered: 2499
/content/drive/MyDrive/palm_video_project/video_frame_selection/Normal/ID890/frame_02.jpg
/content/drive/MyDrive/palm_video_project/video_frame_selection/Normal/ID890/frame_03.jpg
/content/drive/MyDrive/palm_video_project/video_frame_selection/Normal/ID890/frame_01.jpg
/content/drive/MyDrive/palm_video_project/video_frame_selection/Normal/ID894/frame_01.jpg
/content/drive/MyDrive/palm_video_project/video_frame_selection/Normal/ID894/frame_02.jpg
/content/drive/MyDrive/palm_video_project/video_frame_selection/Normal/ID894/frame_03.jpg
/content/drive/MyDrive/palm_video_project/video_frame_selection/Normal/ID891/frame_02.jpg
/content/drive/MyDrive/palm_video_project/video_frame_selection/Normal/ID891/frame_01.jpg
/content/drive/MyDrive/palm_video_project/video_frame_selection/Normal/ID891/frame_03.jpg
/content/drive/MyDrive/palm_video_project/video_frame_selection/Normal/ID892/frame_01.jpg
/content/drive/MyDrive/palm_video_project/video_frame_selectio

In [6]:

import os
import pandas as pd
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/palm_video_project"
ORIGINAL_ROOT = os.path.join(PROJECT_ROOT, "video_frame_selection")

OUTPUT_ROOT = os.path.join(
    PROJECT_ROOT, "experiment_all_original_three_classes"
)
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Exact aligned patient IDs from the nail branch
split_ids = {
    "train": set(train_aligned["ID"].map(clean_id)),
    "validation": set(val_aligned["ID"].map(clean_id)),
    "test": set(test_aligned["ID"].map(clean_id)),
}

# Verify the aligned splits do not overlap
assert split_ids["train"].isdisjoint(split_ids["validation"])
assert split_ids["train"].isdisjoint(split_ids["test"])
assert split_ids["validation"].isdisjoint(split_ids["test"])

# Original folder names and binary labels
severity_mapping = {
    "Normal": ("Normal", 0),
    "Leve": ("Leve", 1),
    "Moderado": ("Moderado", 1),
    "Moderada": ("Moderada", 1),
}

# Map every aligned ID to exactly one split
id_to_split = {}
for split_name, ids in split_ids.items():
    for patient_id in ids:
        id_to_split[patient_id] = split_name

records = []
unmatched_ids = set(id_to_split)
unmatched_folders = []

for class_dir in Path(ORIGINAL_ROOT).iterdir():
    if not class_dir.is_dir():
        continue

    folder_class = class_dir.name

    if folder_class not in severity_mapping:
        unmatched_folders.append(folder_class)
        continue

    severity_class, anemia_binary = severity_mapping[folder_class]

    for patient_dir in class_dir.iterdir():
        if not patient_dir.is_dir():
            continue

        patient_id = clean_id(patient_dir.name)

        # Ignore patients not included in any aligned nail split
        if patient_id not in id_to_split:
            continue

        unmatched_ids.discard(patient_id)

        for image_path in patient_dir.rglob("*"):
            if not image_path.is_file():
                continue
            if image_path.suffix.lower() not in {
                ".jpg", ".jpeg", ".png", ".bmp", ".webp"
            }:
                continue

            records.append({
                "ID": patient_id,
                "Severity_Class": severity_class,
                "Anemia_Binary": anemia_binary,
                "image_path": str(image_path),
                "source_type": "real_original",
                "split": id_to_split[patient_id],
            })

original_manifest = pd.DataFrame(records)

if original_manifest.empty:
    raise ValueError(
        "No original frames matched the aligned IDs. "
        "Check patient ID formats and folder names."
    )

# Check each patient has only one original severity label
label_counts = (
    original_manifest.groupby("ID")["Severity_Class"].nunique()
)
conflicting_ids = label_counts[label_counts > 1].index.tolist()

if conflicting_ids:
    raise ValueError(
        "Some patients appear under multiple severity folders: "
        f"{conflicting_ids[:20]}"
    )

# Save each split separately
for split_name in ["train", "validation", "test"]:
    split_df = (
        original_manifest[original_manifest["split"] == split_name]
        .copy()
        .reset_index(drop=True)
    )

    output_path = os.path.join(
        OUTPUT_ROOT,
        f"palm_{split_name}_original_manifest.csv"
    )
    split_df.to_csv(output_path, index=False)

    print(f"\n{split_name.upper()}")
    print("Frames:", len(split_df))
    print("Unique patients:", split_df["ID"].nunique())
    print("Patients by severity:")
    print(
        split_df.drop_duplicates("ID")["Severity_Class"]
        .value_counts().to_string()
    )
    print("Saved:", output_path)

print("\nTOTAL ORIGINAL FRAMES:", len(original_manifest))
print("TOTAL UNIQUE PATIENTS:", original_manifest["ID"].nunique())
print("Aligned IDs with no original folder:", len(unmatched_ids))
print("Examples of missing IDs:", sorted(unmatched_ids)[:30])
print("Unrecognized class folders:", unmatched_folders)

print("\nFrame counts by split and severity:")
print(pd.crosstab(
    original_manifest["split"],
    original_manifest["Severity_Class"]
))

print("\nOriginal source types:")
print(original_manifest["source_type"].value_counts())



TRAIN
Frames: 1572
Unique patients: 524
Patients by severity:
Severity_Class
Normal      345
Leve        130
Moderado     49
Saved: /content/drive/MyDrive/palm_video_project/experiment_all_original_three_classes/palm_train_original_manifest.csv

VALIDATION
Frames: 276
Unique patients: 92
Patients by severity:
Severity_Class
Normal      61
Leve        22
Moderado     9
Saved: /content/drive/MyDrive/palm_video_project/experiment_all_original_three_classes/palm_validation_original_manifest.csv

TEST
Frames: 465
Unique patients: 155
Patients by severity:
Severity_Class
Normal      102
Leve         39
Moderado     14
Saved: /content/drive/MyDrive/palm_video_project/experiment_all_original_three_classes/palm_test_original_manifest.csv

TOTAL ORIGINAL FRAMES: 2313
TOTAL UNIQUE PATIENTS: 771
Aligned IDs with no original folder: 0
Examples of missing IDs: []
Unrecognized class folders: []

Frame counts by split and severity:
Severity_Class  Leve  Moderado  Normal
split                         

In [7]:

# Verify all image paths point to original frames
assert original_manifest["image_path"].map(os.path.isfile).all()
assert original_manifest["image_path"].str.contains(
    "/video_frame_selection/"
).all()

assert set(original_manifest["source_type"]) == {"real_original"}

for split_name, expected_ids in split_ids.items():
    split_df = original_manifest[
        original_manifest["split"] == split_name
    ]

    actual_ids = set(split_df["ID"].unique())

    print(
        f"{split_name}: "
        f"{len(actual_ids)} matched patients / "
        f"{len(expected_ids)} aligned patients"
    )

    assert actual_ids.issubset(expected_ids)

# Confirm there is no patient overlap between the actual datasets
actual_split_ids = {
    name: set(original_manifest.loc[
        original_manifest["split"] == name, "ID"
    ].unique())
    for name in ["train", "validation", "test"]
}

assert actual_split_ids["train"].isdisjoint(
    actual_split_ids["validation"]
)
assert actual_split_ids["train"].isdisjoint(
    actual_split_ids["test"]
)
assert actual_split_ids["validation"].isdisjoint(
    actual_split_ids["test"]
)

print("\nValidation passed: original images and aligned patient splits verified.")


train: 524 matched patients / 524 aligned patients
validation: 92 matched patients / 92 aligned patients
test: 155 matched patients / 155 aligned patients

Validation passed: original images and aligned patient splits verified.


In [8]:
import os
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/palm_video_project"

EXPERIMENT_ROOT = os.path.join(
    PROJECT_ROOT, "experiment_all_original_three_classes"
)

MANIFEST_PATHS = {
    "train": os.path.join(
        EXPERIMENT_ROOT, "palm_train_original_manifest.csv"
    ),
    "validation": os.path.join(
        EXPERIMENT_ROOT, "palm_validation_original_manifest.csv"
    ),
    "test": os.path.join(
        EXPERIMENT_ROOT, "palm_test_original_manifest.csv"
    ),
}

manifests = {
    split: pd.read_csv(path)
    for split, path in MANIFEST_PATHS.items()
}

train_df = manifests["train"].copy()
val_df = manifests["validation"].copy()
test_df = manifests["test"].copy()

required_columns = {
    "ID", "Severity_Class", "Anemia_Binary",
    "image_path", "source_type", "split"
}

for split, df in manifests.items():
    missing_columns = required_columns - set(df.columns)
    assert not missing_columns, (
        f"{split} manifest is missing columns: {missing_columns}"
    )

    assert df["image_path"].map(os.path.isfile).all(), (
        f"Some image paths are missing in {split}"
    )

    assert set(df["source_type"].dropna().unique()) == {
        "real_original"
    }, f"Unexpected image source in {split}"

    assert df["split"].eq(split).all(), (
        f"Unexpected split label in {split}"
    )

    assert df["Anemia_Binary"].isin([0, 1]).all()

# Confirm no patient occurs in more than one split.
train_ids = set(train_df["ID"].astype(str))
val_ids = set(val_df["ID"].astype(str))
test_ids = set(test_df["ID"].astype(str))

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("Manifest verification passed.\n")

for split, df in manifests.items():
    print(
        f"{split.upper():10s} | "
        f"Frames: {len(df):4d} | "
        f"Patients: {df['ID'].nunique():3d}"
    )

print("\nBinary patient counts:")
for split, df in manifests.items():
    patient_labels = df.drop_duplicates("ID")
    print(f"\n{split.upper()}")
    print(
        patient_labels["Anemia_Binary"]
        .value_counts()
        .sort_index()
        .rename(index={0: "Normal (0)", 1: "Anemia (1)"})
        .to_string()
    )

Manifest verification passed.

TRAIN      | Frames: 1572 | Patients: 524
VALIDATION | Frames:  276 | Patients:  92
TEST       | Frames:  465 | Patients: 155

Binary patient counts:

TRAIN
Anemia_Binary
Normal (0)    345
Anemia (1)    179

VALIDATION
Anemia_Binary
Normal (0)    61
Anemia (1)    31

TEST
Anemia_Binary
Normal (0)    102
Anemia (1)     53


In [9]:
import random
import copy
import json
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image, ImageFile

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [10]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0  # Safer for Google Colab

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=8),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.08,
    ),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])


class PalmFrameDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        image_path = row["image_path"]
        label = int(row["Anemia_Binary"])

        with Image.open(image_path) as image:
            image = image.convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return (
            image,
            torch.tensor(label, dtype=torch.float32),
            str(row["ID"]),
            image_path,
        )


train_dataset = PalmFrameDataset(train_df, train_transform)
val_dataset = PalmFrameDataset(val_df, eval_transform)
test_dataset = PalmFrameDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

# Inspect one batch.
images, labels, patient_ids, image_paths = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Example labels:", labels[:10].tolist())
print("Example patient IDs:", list(patient_ids[:5]))

Train batches: 50
Validation batches: 9
Test batches: 15
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Example labels: [0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0]
Example patient IDs: ['ID136', 'ID676', 'ID845', 'ID247', 'ID518']


In [11]:
# Calculate positive-class weight from TRAINING frames only.
negative_count = int((train_df["Anemia_Binary"] == 0).sum())
positive_count = int((train_df["Anemia_Binary"] == 1).sum())

pos_weight_value = negative_count / max(positive_count, 1)
pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=DEVICE,
)

print("Training normal frames:", negative_count)
print("Training anemia frames:", positive_count)
print("Positive class weight:", round(pos_weight_value, 4))

# Load ImageNet-pretrained EfficientNet-B0.
try:
    weights = models.EfficientNet_B0_Weights.DEFAULT
    model = models.efficientnet_b0(weights=weights)
except Exception as e:
    print("Could not load pretrained weights:", repr(e))
    print("Trying to load model without pretrained weights.")
    model = models.efficientnet_b0(weights=None)

# Replace the final classification layer with a single binary output.
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 1)

model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print("\nModel ready.")
print("Final classifier:", model.classifier)

Training normal frames: 1035
Training anemia frames: 537
Positive class weight: 1.9274
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 171MB/s]


Model ready.
Final classifier: Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1, bias=True)
)


In [12]:
# Freeze the EfficientNet feature extractor.
for param in model.features.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

EPOCHS_HEAD = 5
LEARNING_RATE_HEAD = 1e-3

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE_HEAD,
    weight_decay=1e-4,
)


def run_training_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    total = 0

    for images, labels, _, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True).view(-1, 1)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size

    return running_loss / max(total, 1)


@torch.no_grad()
def calculate_loss(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    total = 0

    for images, labels, _, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True).view(-1, 1)

        logits = model(images)
        loss = criterion(logits, labels)

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size

    return running_loss / max(total, 1)


head_history = []

for epoch in range(1, EPOCHS_HEAD + 1):
    start_time = time.time()

    train_loss = run_training_epoch(
        model, train_loader, optimizer, criterion
    )
    val_loss = calculate_loss(model, val_loader, criterion)

    elapsed = time.time() - start_time

    row = {
        "phase": "head",
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": val_loss,
        "seconds": elapsed,
    }
    head_history.append(row)

    print(
        f"Head epoch {epoch:02d}/{EPOCHS_HEAD} | "
        f"train loss: {train_loss:.4f} | "
        f"validation loss: {val_loss:.4f} | "
        f"time: {elapsed:.1f}s"
    )

head_history_df = pd.DataFrame(head_history)
display(head_history_df)

Head epoch 01/5 | train loss: 0.8879 | validation loss: 0.8400 | time: 711.2s
Head epoch 02/5 | train loss: 0.8551 | validation loss: 0.8167 | time: 55.9s
Head epoch 03/5 | train loss: 0.8307 | validation loss: 0.8244 | time: 55.1s
Head epoch 04/5 | train loss: 0.8270 | validation loss: 0.8380 | time: 55.0s
Head epoch 05/5 | train loss: 0.8084 | validation loss: 0.8296 | time: 55.0s


,phase,epoch,train_loss,validation_loss,seconds
0,head,1,0.887851,0.840008,711.184417
1,head,2,0.855052,0.816666,55.862686
2,head,3,0.830729,0.824360,55.075951
3,head,4,0.826968,0.837966,54.964953
4,head,5,0.808388,0.829639,55.019956


In [13]:
# Unfreeze all model parameters.
for param in model.parameters():
    param.requires_grad = True

EPOCHS_FINE = 15
LEARNING_RATE_FINE = 1e-4

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE_FINE,
    weight_decay=1e-4,
)

# Reduce learning rate if validation loss stops improving.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

BEST_MODEL_PATH = os.path.join(
    EXPERIMENT_ROOT, "efficientnet_b0_original_best.pth"
)

best_val_loss = float("inf")
best_epoch = None
patience = 5
epochs_without_improvement = 0
fine_history = []

for epoch in range(1, EPOCHS_FINE + 1):
    start_time = time.time()

    train_loss = run_training_epoch(
        model, train_loader, optimizer, criterion
    )
    val_loss = calculate_loss(model, val_loader, criterion)

    scheduler.step(val_loss)

    improved = val_loss < best_val_loss

    if improved:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "validation_loss": val_loss,
                "image_size": IMAGE_SIZE,
                "model_name": "efficientnet_b0",
                "label_mapping": {
                    "Normal": 0,
                    "Leve": 1,
                    "Moderado": 1,
                    "Moderada": 1,
                },
            },
            BEST_MODEL_PATH,
        )
    else:
        epochs_without_improvement += 1

    elapsed = time.time() - start_time
    current_lr = optimizer.param_groups[0]["lr"]

    fine_history.append({
        "phase": "fine_tune",
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": val_loss,
        "learning_rate": current_lr,
        "improved": improved,
        "seconds": elapsed,
    })

    print(
        f"Fine-tune epoch {epoch:02d}/{EPOCHS_FINE} | "
        f"train loss: {train_loss:.4f} | "
        f"validation loss: {val_loss:.4f} | "
        f"lr: {current_lr:.2e} | "
        f"{'BEST' if improved else ''}"
    )

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch}.")
        break

print("\nBest epoch:", best_epoch)
print("Best validation loss:", round(best_val_loss, 4))
print("Saved model:", BEST_MODEL_PATH)

history_df = pd.concat(
    [head_history_df, pd.DataFrame(fine_history)],
    ignore_index=True,
)

history_path = os.path.join(
    EXPERIMENT_ROOT, "efficientnet_training_history.csv"
)
history_df.to_csv(history_path, index=False)

print("Saved training history:", history_path)

Fine-tune epoch 01/15 | train loss: 0.7611 | validation loss: 0.8394 | lr: 1.00e-04 | BEST
Fine-tune epoch 02/15 | train loss: 0.6530 | validation loss: 0.8214 | lr: 1.00e-04 | BEST
Fine-tune epoch 03/15 | train loss: 0.5235 | validation loss: 0.9813 | lr: 1.00e-04 | 
Fine-tune epoch 04/15 | train loss: 0.4126 | validation loss: 1.0771 | lr: 1.00e-04 | 
Fine-tune epoch 05/15 | train loss: 0.3033 | validation loss: 1.3043 | lr: 5.00e-05 | 
Fine-tune epoch 06/15 | train loss: 0.2010 | validation loss: 1.3751 | lr: 5.00e-05 | 
Fine-tune epoch 07/15 | train loss: 0.1893 | validation loss: 1.4122 | lr: 5.00e-05 | 
Early stopping at epoch 7.

Best epoch: 2
Best validation loss: 0.8214
Saved model: /content/drive/MyDrive/palm_video_project/experiment_all_original_three_classes/efficientnet_b0_original_best.pth
Saved training history: /content/drive/MyDrive/palm_video_project/experiment_all_original_three_classes/efficientnet_training_history.csv


In [14]:
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded checkpoint from epoch:", checkpoint["epoch"])
print("Checkpoint validation loss:", checkpoint["validation_loss"])


@torch.no_grad()
def predict_frames(model, loader):
    model.eval()
    records = []

    for images, labels, patient_ids, image_paths in loader:
        images = images.to(DEVICE, non_blocking=True)

        logits = model(images).view(-1)
        probabilities = torch.sigmoid(logits).cpu().numpy()

        labels = labels.numpy().astype(int).reshape(-1)

        for patient_id, image_path, label, probability in zip(
            patient_ids,
            image_paths,
            labels,
            probabilities,
        ):
            records.append({
                "ID": str(patient_id),
                "image_path": image_path,
                "true_label": int(label),
                "anemia_probability": float(probability),
            })

    return pd.DataFrame(records)


val_frame_predictions = predict_frames(model, val_loader)
test_frame_predictions = predict_frames(model, test_loader)

val_frame_path = os.path.join(
    EXPERIMENT_ROOT, "validation_frame_predictions.csv"
)
test_frame_path = os.path.join(
    EXPERIMENT_ROOT, "test_frame_predictions.csv"
)

val_frame_predictions.to_csv(val_frame_path, index=False)
test_frame_predictions.to_csv(test_frame_path, index=False)

print("Validation frame predictions:", len(val_frame_predictions))
print("Test frame predictions:", len(test_frame_predictions))
print("Saved:", val_frame_path)
print("Saved:", test_frame_path)

display(test_frame_predictions.head())

Loaded checkpoint from epoch: 2
Checkpoint validation loss: 0.8213913026063339
Validation frame predictions: 276
Test frame predictions: 465
Saved: /content/drive/MyDrive/palm_video_project/experiment_all_original_three_classes/validation_frame_predictions.csv
Saved: /content/drive/MyDrive/palm_video_project/experiment_all_original_three_classes/test_frame_predictions.csv


,ID,image_path,true_label,anemia_probability
0,ID896,/content/drive/MyDrive/palm_video_project/vide...,1,0.187655
1,ID896,/content/drive/MyDrive/palm_video_project/vide...,1,0.169293
2,ID896,/content/drive/MyDrive/palm_video_project/vide...,1,0.106505
3,ID839,/content/drive/MyDrive/palm_video_project/vide...,1,0.357616
4,ID839,/content/drive/MyDrive/palm_video_project/vide...,1,0.759267


In [15]:
def aggregate_patient_predictions(frame_predictions, threshold=0.5):
    patient_df = (
        frame_predictions
        .groupby("ID", as_index=False)
        .agg(
            true_label=("true_label", "first"),
            anemia_probability=("anemia_probability", "mean"),
            frame_count=("image_path", "count"),
        )
    )

    patient_df["predicted_label"] = (
        patient_df["anemia_probability"] >= threshold
    ).astype(int)

    return patient_df


def evaluate_patient_predictions(patient_df, split_name):
    y_true = patient_df["true_label"].astype(int)
    y_pred = patient_df["predicted_label"].astype(int)
    y_prob = patient_df["anemia_probability"].astype(float)

    results = {
        "split": split_name,
        "patients": int(len(patient_df)),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall_sensitivity": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "specificity": (
            (confusion_matrix(y_true, y_pred, labels=[0, 1])[0, 0])
            / max(
                confusion_matrix(y_true, y_pred, labels=[0, 1])[0].sum(),
                1,
            )
        ),
        "roc_auc": (
            roc_auc_score(y_true, y_prob)
            if y_true.nunique() == 2
            else np.nan
        ),
    }

    print(f"\n{'=' * 55}")
    print(f"{split_name.upper()} — PATIENT-LEVEL EVALUATION")
    print(f"{'=' * 55}")
    print("Patients:", results["patients"])
    print("Accuracy:", round(results["accuracy"], 4))
    print("Precision:", round(results["precision"], 4))
    print("Sensitivity / Recall:", round(results["recall_sensitivity"], 4))
    print("Specificity:", round(results["specificity"], 4))
    print("F1-score:", round(results["f1"], 4))
    print("ROC-AUC:", round(results["roc_auc"], 4)
          if not np.isnan(results["roc_auc"]) else "Not available")

    print("\nConfusion matrix (rows=true, columns=predicted):")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=["Normal", "Anemia"],
            zero_division=0,
        )
    )

    return results


# Validation is used for inspection/model selection.
val_patient_predictions = aggregate_patient_predictions(
    val_frame_predictions
)
val_metrics = evaluate_patient_predictions(
    val_patient_predictions, "validation"
)

# Final test evaluation: run after training and checkpoint selection.
test_patient_predictions = aggregate_patient_predictions(
    test_frame_predictions
)
test_metrics = evaluate_patient_predictions(
    test_patient_predictions, "test"
)

val_patient_predictions.to_csv(
    os.path.join(EXPERIMENT_ROOT, "validation_patient_predictions.csv"),
    index=False,
)
test_patient_predictions.to_csv(
    os.path.join(EXPERIMENT_ROOT, "test_patient_predictions.csv"),
    index=False,
)

metrics_path = os.path.join(
    EXPERIMENT_ROOT, "patient_level_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(
        {
            "validation": val_metrics,
            "test": test_metrics,
            "threshold": 0.5,
            "aggregation": "mean frame probability per patient",
            "checkpoint_epoch": int(checkpoint["epoch"]),
        },
        f,
        indent=2,
    )

print("\nSaved patient-level metrics:", metrics_path)


VALIDATION — PATIENT-LEVEL EVALUATION
Patients: 92
Accuracy: 0.663
Precision: 0.5
Sensitivity / Recall: 0.5484
Specificity: 0.7213
F1-score: 0.5231
ROC-AUC: 0.7292

Confusion matrix (rows=true, columns=predicted):
[[44 17]
 [14 17]]

Classification report:
              precision    recall  f1-score   support

      Normal       0.76      0.72      0.74        61
      Anemia       0.50      0.55      0.52        31

    accuracy                           0.66        92
   macro avg       0.63      0.63      0.63        92
weighted avg       0.67      0.66      0.67        92


TEST — PATIENT-LEVEL EVALUATION
Patients: 155
Accuracy: 0.6968
Precision: 0.5517
Sensitivity / Recall: 0.6038
Specificity: 0.7451
F1-score: 0.5766
ROC-AUC: 0.6853

Confusion matrix (rows=true, columns=predicted):
[[76 26]
 [21 32]]

Classification report:
              precision    recall  f1-score   support

      Normal       0.78      0.75      0.76       102
      Anemia       0.55      0.60      0.58      

In [16]:
import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

EXPERIMENT_ROOT = (
    "/content/drive/MyDrive/palm_video_project/"
    "experiment_all_original_three_classes"
)

val_patients = pd.read_csv(
    os.path.join(
        EXPERIMENT_ROOT,
        "validation_patient_predictions.csv"
    )
)

test_patients = pd.read_csv(
    os.path.join(
        EXPERIMENT_ROOT,
        "test_patient_predictions.csv"
    )
)


def calculate_threshold_metrics(df, threshold):
    y_true = df["true_label"].astype(int).to_numpy()
    y_prob = df["anemia_probability"].astype(float).to_numpy()
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "sensitivity": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "specificity": tn / max(tn + fp, 1),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
    }


thresholds = np.round(np.arange(0.20, 0.81, 0.05), 2)

val_threshold_results = pd.DataFrame([
    calculate_threshold_metrics(val_patients, threshold)
    for threshold in thresholds
])

test_threshold_results = pd.DataFrame([
    calculate_threshold_metrics(test_patients, threshold)
    for threshold in thresholds
])

print("VALIDATION — use this table to choose a threshold")
display(val_threshold_results.round(4))

print("TEST — descriptive comparison only; do not select a threshold here")
display(test_threshold_results.round(4))

VALIDATION — use this table to choose a threshold


,threshold,accuracy,precision,sensitivity,specificity,f1,TP,TN,FP,FN
0,0.20,0.5761,0.4412,0.9677,0.3770,0.6061,30,23,38,1
1,0.25,0.6522,0.4915,0.9355,0.5082,0.6444,29,31,30,2
2,0.30,0.6522,0.4909,0.8710,0.5410,0.6279,27,33,28,4
3,0.35,0.6630,0.5000,0.8065,0.5902,0.6173,25,36,25,6
4,0.40,0.6413,0.4762,0.6452,0.6393,0.5479,20,39,22,11
5,0.45,0.6522,0.4872,0.6129,0.6721,0.5429,19,41,20,12
6,0.50,0.6630,0.5000,0.5484,0.7213,0.5231,17,44,17,14
7,0.55,0.6413,0.4643,0.4194,0.7541,0.4407,13,46,15,18
8,0.60,0.6413,0.4500,0.2903,0.8197,0.3529,9,50,11,22
9,0.65,0.6522,0.4667,0.2258,0.8689,0.3043,7,53,8,24


TEST — descriptive comparison only; do not select a threshold here


,threshold,accuracy,precision,sensitivity,specificity,f1,TP,TN,FP,FN
0,0.20,0.4968,0.3984,0.9245,0.2745,0.5568,49,28,74,4
1,0.25,0.5419,0.4211,0.9057,0.3529,0.5749,48,36,66,5
2,0.30,0.5548,0.4231,0.8302,0.4118,0.5605,44,42,60,9
3,0.35,0.6065,0.4565,0.7925,0.5098,0.5793,42,52,50,11
4,0.40,0.6323,0.4750,0.7170,0.5882,0.5714,38,60,42,15
5,0.45,0.6645,0.5072,0.6604,0.6667,0.5738,35,68,34,18
6,0.50,0.6968,0.5517,0.6038,0.7451,0.5766,32,76,26,21
7,0.55,0.6516,0.4884,0.3962,0.7843,0.4375,21,80,22,32
8,0.60,0.6387,0.4545,0.2830,0.8235,0.3488,15,84,18,38
9,0.65,0.6194,0.4000,0.2264,0.8235,0.2892,12,84,18,41


In [17]:
import os
import json
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

EXPERIMENT_ROOT = (
    "/content/drive/MyDrive/palm_video_project/"
    "experiment_all_original_three_classes"
)

val_df = pd.read_csv(
    os.path.join(EXPERIMENT_ROOT, "validation_patient_predictions.csv")
)

def threshold_report(df, threshold):
    y_true = df["true_label"].astype(int)
    y_prob = df["anemia_probability"].astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else 0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }

comparison = pd.DataFrame([
    threshold_report(val_df, 0.25),
    threshold_report(val_df, 0.50),
])

display(comparison.round(4))

# Save the candidate operating threshold and its rationale.
threshold_config = {
    "candidate_threshold": 0.25,
    "baseline_threshold": 0.50,
    "selection_split": "validation",
    "selection_rationale": (
        "Highest validation F1-score among thresholds evaluated "
        "from 0.20 to 0.80 in increments of 0.05; "
        "candidate for further validation, not a clinical threshold."
    ),
}

config_path = os.path.join(
    EXPERIMENT_ROOT, "threshold_selection.json"
)

with open(config_path, "w") as f:
    json.dump(threshold_config, f, indent=2)

print("Saved threshold configuration:", config_path)

,threshold,accuracy,precision,sensitivity,specificity,f1,TP,TN,FP,FN
0,0.25,0.6522,0.4915,0.9355,0.5082,0.6444,29,31,30,2
1,0.50,0.6630,0.5000,0.5484,0.7213,0.5231,17,44,17,14


Saved threshold configuration: /content/drive/MyDrive/palm_video_project/experiment_all_original_three_classes/threshold_selection.json


In [18]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageStat, ImageFilter

PROJECT_ROOT = "/content/drive/MyDrive/palm_video_project"

EXPERIMENT_ROOT = os.path.join(
    PROJECT_ROOT,
    "experiment_all_original_three_classes"
)

OUTPUT_ROOT = os.path.join(
    EXPERIMENT_ROOT,
    "quality_scores"
)
os.makedirs(OUTPUT_ROOT, exist_ok=True)

train_df = pd.read_csv(
    os.path.join(EXPERIMENT_ROOT, "palm_train_original_manifest.csv")
)
val_df = pd.read_csv(
    os.path.join(EXPERIMENT_ROOT, "palm_validation_original_manifest.csv")
)
test_df = pd.read_csv(
    os.path.join(EXPERIMENT_ROOT, "palm_test_original_manifest.csv")
)

all_manifest = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

all_manifest = all_manifest.drop_duplicates(
    subset=["image_path"]
).reset_index(drop=True)

assert len(all_manifest) == 2313, (
    f"Expected 2313 original frames, found {len(all_manifest)}"
)

assert all_manifest["image_path"].map(os.path.isfile).all()
assert all_manifest["source_type"].eq("real_original").all()

print("Original frames:", len(all_manifest))
print("Unique patients:", all_manifest["ID"].nunique())
print("Split counts:")
print(all_manifest["split"].value_counts())
print("\nOutput directory:", OUTPUT_ROOT)

Original frames: 2313
Unique patients: 771
Split counts:
split
train         1572
test           465
validation     276
Name: count, dtype: int64

Output directory: /content/drive/MyDrive/palm_video_project/experiment_all_original_three_classes/quality_scores


In [19]:
import warnings

def extract_quality_features(image_path):
    with Image.open(image_path) as img:
        rgb = img.convert("RGB")
        width, height = rgb.size

        rgb_array = np.asarray(rgb, dtype=np.float32) / 255.0

        # Convert RGB to approximate HSV saturation.
        max_channel = rgb_array.max(axis=2)
        min_channel = rgb_array.min(axis=2)
        delta = max_channel - min_channel

        saturation = np.zeros_like(max_channel)
        nonzero = max_channel > 1e-8
        saturation[nonzero] = (
            delta[nonzero] / max_channel[nonzero]
        )

        # Approximate luminance (0-1).
        luminance = (
            0.2126 * rgb_array[:, :, 0]
            + 0.7152 * rgb_array[:, :, 1]
            + 0.0722 * rgb_array[:, :, 2]
        )

        brightness = float(luminance.mean())
        contrast = float(luminance.std())

        highlight_fraction = float(
            np.mean(luminance >= 0.90)
        )
        shadow_fraction = float(
            np.mean(luminance <= 0.10)
        )

        # Variance of Laplacian as a simple sharpness proxy.
        gray = rgb.convert("L")
        laplacian = gray.filter(ImageFilter.FIND_EDGES)
        sharpness = float(
            np.asarray(laplacian, dtype=np.float32).var()
        )

        return {
            "brightness": brightness,
            "contrast": contrast,
            "saturation": float(saturation.mean()),
            "highlight_fraction": highlight_fraction,
            "shadow_fraction": shadow_fraction,
            "sharpness": sharpness,
            "original_width": int(width),
            "original_height": int(height),
            "original_area": int(width * height),
            "aspect_ratio": float(width / height) if height else np.nan,
        }


quality_records = []
errors = []

for idx, row in all_manifest.iterrows():
    image_path = row["image_path"]

    try:
        features = extract_quality_features(image_path)

        quality_records.append({
            "ID": str(row["ID"]),
            "split": row["split"],
            "Severity_Class": row["Severity_Class"],
            "Anemia_Binary": int(row["Anemia_Binary"]),
            "image_path": image_path,
            **features,
        })

    except Exception as exc:
        errors.append({
            "image_path": image_path,
            "error": repr(exc),
        })

    if (idx + 1) % 250 == 0 or idx + 1 == len(all_manifest):
        print(f"Processed {idx + 1}/{len(all_manifest)} frames")

quality_df = pd.DataFrame(quality_records)

quality_path = os.path.join(
    OUTPUT_ROOT,
    "original_frame_quality_features.csv"
)
quality_df.to_csv(quality_path, index=False)

print("\nQuality feature extraction complete.")
print("Successful frames:", len(quality_df))
print("Failed frames:", len(errors))
print("Saved:", quality_path)

if errors:
    error_path = os.path.join(OUTPUT_ROOT, "quality_extraction_errors.csv")
    pd.DataFrame(errors).to_csv(error_path, index=False)
    print("Errors saved:", error_path)

display(quality_df.head())

Processed 250/2313 frames
Processed 500/2313 frames
Processed 750/2313 frames
Processed 1000/2313 frames
Processed 1250/2313 frames
Processed 1500/2313 frames
Processed 1750/2313 frames
Processed 2000/2313 frames
Processed 2250/2313 frames
Processed 2313/2313 frames

Quality feature extraction complete.
Successful frames: 2313
Failed frames: 0
Saved: /content/drive/MyDrive/palm_video_project/experiment_all_original_three_classes/quality_scores/original_frame_quality_features.csv


,ID,split,Severity_Class,Anemia_Binary,image_path,brightness,contrast,saturation,highlight_fraction,shadow_fraction,sharpness,original_width,original_height,original_area,aspect_ratio
0,ID909,train,Leve,1,/content/drive/MyDrive/palm_video_project/vide...,0.585320,0.336830,0.246146,0.107099,0.174371,73.700668,1440,1440,2073600,1.0
1,ID909,train,Leve,1,/content/drive/MyDrive/palm_video_project/vide...,0.595701,0.341729,0.235297,0.146577,0.168207,66.932228,1440,1440,2073600,1.0
2,ID909,train,Leve,1,/content/drive/MyDrive/palm_video_project/vide...,0.588899,0.338270,0.241098,0.114667,0.171485,74.203835,1440,1440,2073600,1.0
3,ID865,train,Leve,1,/content/drive/MyDrive/palm_video_project/vide...,0.549986,0.307236,0.270922,0.001783,0.145273,82.325531,1440,1440,2073600,1.0
4,ID865,train,Leve,1,/content/drive/MyDrive/palm_video_project/vide...,0.550725,0.309554,0.269682,0.002077,0.150421,77.181923,1440,1440,2073600,1.0


In [20]:
quality_columns = [
    "brightness",
    "contrast",
    "saturation",
    "highlight_fraction",
    "shadow_fraction",
    "sharpness",
    "original_width",
    "original_height",
    "original_area",
    "aspect_ratio",
]

patient_quality_summary = (
    quality_df.groupby(
        ["split", "ID", "Severity_Class", "Anemia_Binary"],
        as_index=False
    )
    .agg(
        frame_count=("image_path", "count"),
        **{
            f"{col}_mean": (col, "mean")
            for col in quality_columns
        },
        **{
            f"{col}_std": (col, "std")
            for col in quality_columns
        },
    )
)

summary_path = os.path.join(
    OUTPUT_ROOT,
    "original_patient_quality_summary.csv"
)
patient_quality_summary.to_csv(summary_path, index=False)

print("Patient quality summaries:", len(patient_quality_summary))
print("Saved:", summary_path)
display(patient_quality_summary.head())

Patient quality summaries: 771
Saved: /content/drive/MyDrive/palm_video_project/experiment_all_original_three_classes/quality_scores/original_patient_quality_summary.csv


,split,ID,Severity_Class,Anemia_Binary,frame_count,brightness_mean,contrast_mean,saturation_mean,highlight_fraction_mean,shadow_fraction_mean,...,brightness_std,contrast_std,saturation_std,highlight_fraction_std,shadow_fraction_std,sharpness_std,original_width_std,original_height_std,original_area_std,aspect_ratio_std
0,test,ID003,Normal,0,3,0.558578,0.288109,0.324513,0.000426,0.138418,...,0.009126,0.003270,0.001719,0.000463,0.002305,16.354749,0.0,0.0,0.0,0.0
1,test,ID024,Leve,1,3,0.583181,0.294746,0.254667,0.007037,0.150277,...,0.005477,0.002899,0.006093,0.002510,0.001098,7.798802,0.0,0.0,0.0,0.0
2,test,ID032,Normal,0,3,0.594497,0.305208,0.257224,0.025707,0.180049,...,0.004125,0.001823,0.002207,0.002090,0.001764,10.746609,0.0,0.0,0.0,0.0
3,test,ID033,Leve,1,3,0.485744,0.294398,0.277505,0.004397,0.146287,...,0.001690,0.001675,0.002238,0.003064,0.000741,12.487955,0.0,0.0,0.0,0.0
4,test,ID044,Moderado,1,3,0.538667,0.318932,0.257920,0.008295,0.198032,...,0.001461,0.000908,0.002387,0.000208,0.000708,5.705345,0.0,0.0,0.0,0.0


In [21]:
val_probs_path = os.path.join(
    EXPERIMENT_ROOT,
    "validation_frame_predictions.csv"
)
test_probs_path = os.path.join(
    EXPERIMENT_ROOT,
    "test_frame_predictions.csv"
)

val_probs = pd.read_csv(val_probs_path)
test_probs = pd.read_csv(test_probs_path)

# For the training set, generate probabilities using the saved best checkpoint.
# This is useful for descriptive analysis, but these are in-sample predictions.
checkpoint = torch.load(
    os.path.join(EXPERIMENT_ROOT, "efficientnet_b0_original_best.pth"),
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

train_loader_eval = DataLoader(
    PalmFrameDataset(train_df, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

train_probs = predict_frames(model, train_loader_eval)

train_probs["split"] = "train"
val_probs["split"] = "validation"
test_probs["split"] = "test"

all_probabilities = pd.concat(
    [train_probs, val_probs, test_probs],
    ignore_index=True
)

# Check that every image path maps to exactly one probability.
assert not all_probabilities["image_path"].duplicated().any(), (
    "Duplicate image paths found in probability tables"
)

quality_with_probs = quality_df.merge(
    all_probabilities[
        ["ID", "split", "image_path", "true_label", "anemia_probability"]
    ],
    on=["ID", "split", "image_path"],
    how="left",
    validate="one_to_one",
)

print("Frames with quality scores:", len(quality_df))
print(
    "Frames with probabilities:",
    quality_with_probs["anemia_probability"].notna().sum()
)

missing_probabilities = quality_with_probs[
    quality_with_probs["anemia_probability"].isna()
]

print("Missing probability rows:", len(missing_probabilities))

if len(missing_probabilities) > 0:
    display(missing_probabilities[["ID", "split", "image_path"]].head(10))
    raise ValueError(
        "Some quality-feature rows did not match model probabilities. "
        "Resolve these before exporting."
    )

probability_output_path = os.path.join(
    OUTPUT_ROOT,
    "original_frame_quality_scores_and_probabilities.csv"
)
quality_with_probs.to_csv(probability_output_path, index=False)

print("\nSaved combined frame-level data:")
print(probability_output_path)

display(
    quality_with_probs[
        [
            "ID",
            "split",
            "Severity_Class",
            "Anemia_Binary",
            "image_path",
            "brightness",
            "contrast",
            "saturation",
            "highlight_fraction",
            "shadow_fraction",
            "sharpness",
            "anemia_probability",
        ]
    ].head()
)

Frames with quality scores: 2313
Frames with probabilities: 2313
Missing probability rows: 0

Saved combined frame-level data:
/content/drive/MyDrive/palm_video_project/experiment_all_original_three_classes/quality_scores/original_frame_quality_scores_and_probabilities.csv


,ID,split,Severity_Class,Anemia_Binary,image_path,brightness,contrast,saturation,highlight_fraction,shadow_fraction,sharpness,anemia_probability
0,ID909,train,Leve,1,/content/drive/MyDrive/palm_video_project/vide...,0.585320,0.336830,0.246146,0.107099,0.174371,73.700668,0.858118
1,ID909,train,Leve,1,/content/drive/MyDrive/palm_video_project/vide...,0.595701,0.341729,0.235297,0.146577,0.168207,66.932228,0.914371
2,ID909,train,Leve,1,/content/drive/MyDrive/palm_video_project/vide...,0.588899,0.338270,0.241098,0.114667,0.171485,74.203835,0.912569
3,ID865,train,Leve,1,/content/drive/MyDrive/palm_video_project/vide...,0.549986,0.307236,0.270922,0.001783,0.145273,82.325531,0.829598
4,ID865,train,Leve,1,/content/drive/MyDrive/palm_video_project/vide...,0.550725,0.309554,0.269682,0.002077,0.150421,77.181923,0.806380


In [22]:
feature_definitions = {
    "brightness": "Mean approximate luminance, scaled from 0 to 1.",
    "contrast": "Standard deviation of approximate luminance.",
    "saturation": "Mean approximate HSV saturation, scaled from 0 to 1.",
    "highlight_fraction": "Fraction of pixels with luminance >= 0.90.",
    "shadow_fraction": "Fraction of pixels with luminance <= 0.10.",
    "sharpness": (
        "Variance of an edge-filtered grayscale image; "
        "a simple sharpness proxy, not a calibrated blur score."
    ),
    "original_width": "Original image width in pixels.",
    "original_height": "Original image height in pixels.",
    "original_area": "Original width multiplied by original height.",
    "aspect_ratio": "Original width divided by original height.",
    "anemia_probability": (
        "EfficientNet-B0 sigmoid probability for the binary anemia class."
    ),
    "Anemia_Binary": "Ground-truth binary label: Normal=0, anemia=1.",
    "split": "Patient-level train, validation, or test assignment.",
}

definitions_path = os.path.join(
    OUTPUT_ROOT,
    "original_quality_feature_definitions.json"
)

with open(definitions_path, "w") as f:
    json.dump(feature_definitions, f, indent=2)

print("Saved definitions:", definitions_path)

Saved definitions: /content/drive/MyDrive/palm_video_project/experiment_all_original_three_classes/quality_scores/original_quality_feature_definitions.json
